# Curso: Data Mining

**Grupo:** Mineros

**Integrantes:** 
- Delgado Santana, Francisco Luis
- Pando Cabezas, Nicole Rashel
- Rua Pomahuacre, Brayan Anderson

---

<a name='problema'></a>

### <font color="#5B9BD5">Problema aplicado</font>

MINEDU, DRE y UGEL necesitan reconocer perfiles de locales educativos según la cantidad y operatividad de sus recursos tecnológicos, sus líneas de internet contratadas y la cobertura móvil territorial disponible para priorizar diagnósticos e intervenciones.

---

<a name='org-usuario'></a>

### <font color="#5B9BD5">Organización y usuario</font>

MINEDU, a través de la Unidad de Estadística Educativa, junto con las DRE y UGEL son las organizaciones que usarían este análisis. Se benefician porque son las instancias responsables de supervisar directamente los locales educativos de su jurisdicción.

---

<a name='objetivo'></a>

### <font color="#5B9BD5">Objetivo de análisis</font>

Identificar perfiles de locales educativos con condiciones similares de infraestructura digital para reconocer grupos prioritarios de atención.

---

<a name='fuentes'></a>

### <font color="#5B9BD5">Fuentes de datos</font>

#### 4.1 Padrón de Instituciones Educativas (Padlocal)
**Institución:** MINEDU, Unidad de Estadística Educativa

Registro censal de los locales educativos del Perú, con su ubicación geográfica y estado del local. Identifica y georreferencia cada local mediante `CODLOCAL`.

**Enlaces:** [Página de descarga](https://escale.minedu.gob.pe/uee/-/document_library_display/GMv7/view/10632985/73261) · [Descarga directa](https://escale.minedu.gob.pe/documents/10156/10632985/00_Padlocal.zip)

**Variables:** `CODLOCAL`, departamento, provincia, distrito, centro poblado, código de centro poblado, área, gestión, estado del local.

**Acceso:** descarga del ZIP desde la ficha del recurso en ESCALE; el `.dbf` está dentro.

---

#### 4.2 Recursos Tecnológicos (P91)
**Institución:** MINEDU, Unidad de Estadística Educativa

Inventario de equipos tecnológicos por local (PCs, laptops, tablets, proyectores, entre otros), con cantidad total, cantidad operativa, estado de conservación y antigüedad.

**Enlaces:** [Página de descarga](https://escale.minedu.gob.pe/uee/-/document_library_display/GMv7/view/10632985/73279) · [Descarga directa](https://escale.minedu.gob.pe/documents/10156/10632985/67_loc_p91_rectec.zip)

**Variables:** tipo de recurso, cantidad total, cantidad operativa, estado de conservación, antigüedad.

**Acceso:** descarga del ZIP desde la ficha del recurso en ESCALE; el `.dbf` está dentro.

---

#### 4.3 Líneas de Internet (P3230)
**Institución:** MINEDU, Unidad de Estadística Educativa

Registro de las líneas de internet por local: si están activas, tipo de conexión, proveedor, velocidad contratada y velocidad real de bajada/subida.

**Enlaces:** [Página de descarga](https://escale.minedu.gob.pe/uee/-/document_library_display/GMv7/view/10632985/74801) · [Descarga directa](https://escale.minedu.gob.pe/documents/10156/10632985/59_loc_p3230_inter.zip)

**Variables:** línea activa, medio de conexión, proveedor, ancho de banda contratado, velocidad garantizada, velocidad real de bajada/subida, filtro de contenido web.

**Acceso:** descarga del ZIP desde la ficha del recurso en ESCALE; el `.dbf` está dentro.

---

#### 4.4 Cobertura Móvil por Centro Poblado
**Institución:** OSIPTEL

Porcentaje de cobertura móvil (3G/4G/5G) por centro poblado, operadora y tecnología, a nivel territorial, no del local específico.

**Enlace:** [Dataset](https://www.datosabiertos.gob.pe/dataset/porcentaje-de-cobertura-m%C3%B3vil-por-centro-poblado-empresa-operadora-y-tecnolog%C3%ADa)

**Variables:** ubigeo/centro poblado, tecnología, clasificación de cobertura, operadora.

**Acceso:** descarga directa de los archivos Excel del periodo 2025-IV (partes 1 y 2).

## 1. Criterios metodológicos

* Las tablas con varias filas por local se pivotean antes de integrarlas.
* Los faltantes no se convierten en ceros.
* La cobertura de OSIPTEL es contexto del centro poblado, no una medición del local.

| Fuente | Periodo | Granularidad original | Aporte |
|---|---:|---|---|
| MINEDU `Loc_P91_RecTec.dbf` | 2025 | una fila por tipo de recurso y local | cantidad y estado de recursos tecnológicos |
| MINEDU `Loc_P3230_Inter.dbf` | 2025 | una fila por línea de internet y local | actividad, medio, proveedor, velocidades y filtro web |
| MINEDU `Padlocal.dbf` | 2025 | una fila por local | ubicación, centro poblado, ámbito y gestión |
| OSIPTEL cobertura móvil | 2025-IV | una fila por centro poblado | porcentaje del área del CCPP con cobertura móvil |

OSIPTEL reporta el porcentaje de intersección entre la huella de cobertura y el área del centro poblado, no si hay señal dentro de cada colegio. Por eso se usa solo la cobertura garantizada (`CG`) y no `CG+CAR`.

In [1]:
from pathlib import Path
import json, re, unicodedata
import numpy as np
import pandas as pd
import plotly.express as px
from dbfread import DBF
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
px.defaults.template = "plotly_white"

DATOS = Path("datos")
SALIDAS = Path("salidas")
SALIDAS.mkdir(exist_ok=True)

def snake(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode()
    t = t.replace("+", "_mas_")
    return re.sub(r"_+", "_", re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_").lower())

# Guarda un codigo como texto para no perder los ceros de la izquierda
def cod(serie, ancho):
    s = serie.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return s.str.zfill(ancho)

def seleccionar(df, columnas, nombre):
    eliminadas = [c for c in df.columns if c not in columnas]
    print(f"{nombre}: de {df.shape[1]} columnas nos quedamos con {len(columnas)} "
          f"y eliminamos {len(eliminadas)}")
    print(f"\nSE CONSERVAN ({len(columnas)}):")
    print("   " + ", ".join(columnas))
    print(f"\nSE ELIMINAN ({len(eliminadas)}):")
    print("   " + (", ".join(eliminadas) if eliminadas else "ninguna"))
    return df[columnas].copy()

def revisar_vacios(df):
    tabla = pd.DataFrame({
        "columna": df.columns,
        "vacios": df.isna().sum().values,
        "porcentaje_vacios": (100 * df.isna().mean()).values,
        "con_dato": df.notna().sum().values,
    })
    return tabla.sort_values("porcentaje_vacios", ascending=False).reset_index(drop=True)

print("Carpeta de datos:  ", DATOS)
print("Carpeta de salidas:", SALIDAS)


Carpeta de datos:   datos
Carpeta de salidas: salidas


## 2. Carga de los archivos

Los cuatro archivos se leen sin modificarlos. Los `_raw` quedan como referencia.

In [2]:
def leer_dbf(nombre):
    df = pd.DataFrame(iter(DBF(str(DATOS / nombre), encoding="latin1")))
    df.columns = [snake(c) for c in df.columns]
    return df

recursos_raw = leer_dbf("loc_p91_rectec.dbf")      # cuestionario P91
lineas_raw = leer_dbf("Loc_p3230_inter.dbf")       # cuestionario P3230
padron_raw = leer_dbf("Padlocal.dbf")              # padron de locales

cobertura_raw = pd.concat(
    [pd.read_excel(DATOS / f"osiptel_cobertura_2025IV_parte{i}.xlsx",
                   sheet_name="Listado", dtype={"Ubigeo": "string"}) for i in (1, 2)],
    ignore_index=True,
)
cobertura_raw.columns = [snake(c) for c in cobertura_raw.columns]

FUENTES = [("Recursos tecnologicos", recursos_raw), ("Lineas de internet", lineas_raw),
           ("Padron de locales", padron_raw), ("Cobertura movil", cobertura_raw)]

print("TAMANO DE CADA ARCHIVO")
for nombre, df in FUENTES:
    print(f"  {nombre:24} {len(df):>8,} filas   {df.shape[1]:>2} columnas")


TAMANO DE CADA ARCHIVO
  Recursos tecnologicos     126,460 filas   29 columnas
  Lineas de internet         28,752 filas   16 columnas
  Padron de locales          69,642 filas   19 columnas
  Cobertura movil           108,115 filas   38 columnas


### 2.1 Primeras filas de cada archivo

In [3]:
for nombre, df in FUENTES:
    print(f"\n########## {nombre.upper()} — primeras 3 filas ##########")
    display(df.head(3))



########## RECURSOS TECNOLOGICOS — primeras 3 filas ##########


,codlocal,nroced,cuadro,numero,p91_1,p91_2,p91_3_1,p91_3_21,p91_3_22,p91_3_31,p91_3_32,p91_3_411,p91_3_412,p91_3_413,p91_3_4141,p91_3_4142,p91_3_421,p91_3_422,p91_3_423,p91_3_4241,p91_3_4242,p91_3_431,p91_3_432,p91_3_433,p91_3_4341,p91_3_4342,fec_envio,gestion,anexo
0,000024,11,C091,1,01: PC DE ESCRITORIO,3,3,3,0,2,1,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,2025-09-26 15:04:52,1,0
1,000024,11,C091,9,09: PROYECTOR MULTIMEDIA,1,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2025-09-26 15:04:52,1,0
2,000038,11,C091,1,01: PC DE ESCRITORIO,4,4,4,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,2025-09-25 16:38:29,1,0



########## LINEAS DE INTERNET — primeras 3 filas ##########


,codlocal,nroced,cuadro,numero,p3230_1,p3230_2,p3230_3,p3230_4,p3230_5,p3230_6,p3230_7,p3230_8,p3230_9,p3230_10,fec_envio,anexo
0,000024,11,C3230,1,LINEA 01,1,02,01,01,6.00,70.00,6,6,2,2025-09-26 15:04:52,0
1,000038,11,C3230,1,LINEA 01,1,02,09,02,"1,000.00",100.00,50,100,1,2025-09-25 16:38:29,0
2,000062,11,C3230,1,LINEA 01,1,02,09,03,200.00,100.00,100,200,2,2025-09-12 08:33:43,0



########## PADRON DE LOCALES — primeras 3 filas ##########


,codlocal,dir_cen,localidad,codcp_inei,codccpp,cen_pob,codgeo,d_dpto,d_prov,d_dist,d_region,codooii,d_dreugel,estado,d_estado,cloc_l1,area_loc,ges_loc,tip_envio
0,016100,JIRON TERESA GONZALES DE FANNY 543,NICRUPAMPA,0201050001,129688,CENTENARIO,020105,ANCASH,HUARAZ,INDEPENDENCIA,DRE ANCASH,020001,UGEL HUARAZ,1,Activo,016100,1,A,1.Respondio a FUIE
1,015172,JIRON 28 DE JULIO S/N,HUARUPAMPA,0201010001,114517,HUARUPAMPA,020101,ANCASH,HUARAZ,HUARAZ,DRE ANCASH,020001,UGEL HUARAZ,1,Activo,015172,1,A,1.Respondio a FUIE
2,015186,JIRON AMADEO FIGUEROA S/N,,,610619,LA SOLEDAD,020101,ANCASH,HUARAZ,HUARAZ,DRE ANCASH,020001,UGEL HUARAZ,1,Activo,015186,1,A,1.Respondio a FUIE



########## COBERTURA MOVIL — primeras 3 filas ##########


,ubigeo,departamento,provincia,distrito,centropoblado,clasificacion,lat_y,lon_x,bitel_3g_cg,bitel_3g_cg_mas_car,bitel_4g_cg,bitel_4g_cg_mas_car,bitel_5g_cg,bitel_5g_cg_mas_car,claro_2g_cg,claro_2g_cg_mas_car,claro_3g_cg,claro_3g_cg_mas_car,claro_4g_cg,claro_4g_cg_mas_car,claro_5g_cg,claro_5g_cg_mas_car,entel_2g_cg,entel_2g_cg_mas_car,entel_3g_cg,entel_3g_cg_mas_car,entel_4g_cg,entel_4g_cg_mas_car,entel_5g_cg,entel_5g_cg_mas_car,integratel_2g_cg,integratel_2g_cg_mas_car,integratel_3g_cg,integratel_3g_cg_mas_car,integratel_4g_cg,integratel_4g_cg_mas_car,integratel_5g_cg,integratel_5g_cg_mas_car
0,0101010001,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,CHACHAPOYAS,URBANO,-6.23,-77.87,0.55,0.72,0.46,0.94,0.41,0.67,0.61,1.00,0.37,1.00,0.21,0.99,0.00,0.00,0,0.97,0,1.00,0.01,1.00,0,0.00,0.35,0.97,0.84,0.92,0.99,1.00,0.00,0.00
1,0101010002,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,CACLIC,RURAL,-6.20,-77.90,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.22,0.00,0.00,0.00,0.00,0,1.00,0,1.00,0.00,1.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.29,0.00,0.00
2,0101010003,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,VITALIANO,RURAL,-6.21,-77.87,0.00,0.00,0.00,0.55,0.00,0.00,0.00,1.00,0.00,1.00,0.00,1.00,0.00,0.00,0,1.00,0,1.00,0.00,1.00,0,0.00,0.00,0.92,0.00,0.41,1.00,1.00,0.00,0.00


## 3. Claves de cada fuente

| Fuente | Clave | Una fila es | ¿Agregar antes de unir? |
|---|---|---|---|
| Recursos tecnológicos | `codlocal` + `numero` | un tipo de recurso dentro de un local | **Sí** |
| Líneas de internet | `codlocal` + `numero` | una línea contratada por un local | **Sí** |
| Padrón de locales | `codlocal` | un local educativo | No |
| Cobertura móvil | `ubigeo` | un centro poblado | No |

`codlocal` une las tres fuentes de MINEDU. La cobertura móvil entra por `codcp_inei` del padrón, que es el mismo código INEI que OSIPTEL llama `ubigeo`.

Los códigos se guardan como texto por los ceros a la izquierda: leído como número, `016100` se vuelve `16100` y el cruce falla sin avisar.

In [4]:
print("VERIFICACION DE LAS CLAVES (sobre los datos crudos)\n")
for nombre, df, clave in [("Recursos tecnologicos", recursos_raw, ["codlocal", "numero"]),
                          ("Lineas de internet", lineas_raw, ["codlocal", "numero"]),
                          ("Padron de locales", padron_raw, ["codlocal"]),
                          ("Cobertura movil", cobertura_raw, ["ubigeo"])]:
    print(f"  {nombre:24} clave = {' + '.join(clave)}")
    print(f"  {'':24} filas: {len(df):>8,}   claves distintas: {len(df.drop_duplicates(clave)):>8,}"
          f"   filas repetidas: {df.duplicated(clave).sum()}\n")


VERIFICACION DE LAS CLAVES (sobre los datos crudos)

  Recursos tecnologicos    clave = codlocal + numero
                           filas:  126,460   claves distintas:  126,460   filas repetidas: 0

  Lineas de internet       clave = codlocal + numero
                           filas:   28,752   claves distintas:   28,747   filas repetidas: 5

  Padron de locales        clave = codlocal


                           filas:   69,642   claves distintas:   69,642   filas repetidas: 0

  Cobertura movil          clave = ubigeo


                           filas:  108,115   claves distintas:  108,115   filas repetidas: 0



## 4. Recursos tecnológicos

Archivo `loc_p91_rectec.dbf`. Varias filas por local, una por tipo de recurso.

### 4.1 Selección de columnas

9 de 29. Se eliminan los metadatos del envío y las 15 subdivisiones `p91_3_4xx` del bloque 3.4 del cuestionario, que no se usan en esta entrega.

In [5]:
COLUMNAS_RECURSOS = [
    "codlocal",     # clave: local educativo
    "numero",       # clave: numero de recurso dentro del local
    "p91_1",        # tipo de recurso
    "p91_2",        # cantidad total
    "p91_3_1",      # cantidad operativa
    "p91_3_21",     # operativos en buen estado
    "p91_3_22",     # operativos que necesitan reparacion
    "p91_3_31",     # operativos con menos de 3 anios
    "p91_3_32",     # operativos con mas de 3 anios
]

recursos = seleccionar(recursos_raw, COLUMNAS_RECURSOS, "Recursos tecnologicos")

print("\nPRIMERAS 3 FILAS DESPUES DE SELECCIONAR")
display(recursos.head(3))


Recursos tecnologicos: de 29 columnas nos quedamos con 9 y eliminamos 20

SE CONSERVAN (9):
   codlocal, numero, p91_1, p91_2, p91_3_1, p91_3_21, p91_3_22, p91_3_31, p91_3_32

SE ELIMINAN (20):
   nroced, cuadro, p91_3_411, p91_3_412, p91_3_413, p91_3_4141, p91_3_4142, p91_3_421, p91_3_422, p91_3_423, p91_3_4241, p91_3_4242, p91_3_431, p91_3_432, p91_3_433, p91_3_4341, p91_3_4342, fec_envio, gestion, anexo

PRIMERAS 3 FILAS DESPUES DE SELECCIONAR


,codlocal,numero,p91_1,p91_2,p91_3_1,p91_3_21,p91_3_22,p91_3_31,p91_3_32
0,000024,1,01: PC DE ESCRITORIO,3,3,3,0,2,1
1,000024,9,09: PROYECTOR MULTIMEDIA,1,1,1,0,0,1
2,000038,1,01: PC DE ESCRITORIO,4,4,4,0,0,4


### 4.2 Limpieza

Renombrar, claves a texto, extraer la categoría del recurso (`"01: PC DE ESCRITORIO"` → `pc_escritorio`) y cantidades a número.

In [6]:
recursos = recursos.rename(columns={
    "p91_1": "tipo_recurso_original",
    "p91_2": "cantidad_total",
    "p91_3_1": "cantidad_operativos",
    "p91_3_21": "cantidad_buen_estado",
    "p91_3_22": "cantidad_necesita_reparacion",
    "p91_3_31": "cantidad_menos_3_anios",
    "p91_3_32": "cantidad_mas_3_anios",
})

recursos["codlocal"] = cod(recursos["codlocal"], 6)
recursos["numero_recurso"] = cod(recursos["numero"], 2)

TIPOS_DE_RECURSO = {
    "01": "pc_escritorio", "02": "laptop_convencional", "03": "laptop_xo",
    "04": "servidor", "05": "microservidor", "06": "tablet_aprendo_casa",
    "07": "tablet_pronatel", "08": "tablet_otros", "09": "proyector_multimedia",
    "10": "pizarra_digital", "11": "consola_audio", "12": "auriculares",
    "13": "otro_recurso",
}
recursos["tipo_recurso"] = (recursos["tipo_recurso_original"].astype("string")
                            .str.extract(r"^(\d{2})", expand=False)
                            .map(TIPOS_DE_RECURSO).fillna("sin_dato"))

CANTIDADES = ["cantidad_total", "cantidad_operativos", "cantidad_buen_estado",
              "cantidad_necesita_reparacion", "cantidad_menos_3_anios", "cantidad_mas_3_anios"]
for c in CANTIDADES:
    recursos[c] = pd.to_numeric(recursos[c], errors="coerce")

print("REGLAS DE COHERENCIA (cuantas filas incumplen cada regla)")
calidad_recursos = pd.DataFrame({
    "regla": ["La clave codlocal + numero_recurso esta repetida",
              "La cantidad operativa es mayor que la cantidad total",
              "Buen estado + reparacion no suma la cantidad operativa",
              "Menos de 3 anios + mas de 3 anios no suma la cantidad operativa",
              "Hay cantidades negativas",
              "El tipo de recurso no se pudo traducir"],
    "filas_que_incumplen": [
        recursos.duplicated(["codlocal", "numero_recurso"]).sum(),
        (recursos["cantidad_operativos"] > recursos["cantidad_total"]).sum(),
        ((recursos["cantidad_buen_estado"] + recursos["cantidad_necesita_reparacion"])
         != recursos["cantidad_operativos"]).sum(),
        ((recursos["cantidad_menos_3_anios"] + recursos["cantidad_mas_3_anios"])
         != recursos["cantidad_operativos"]).sum(),
        (recursos[CANTIDADES] < 0).any(axis=1).sum(),
        (recursos["tipo_recurso"] == "sin_dato").sum()],
})
display(calidad_recursos)

print("VALORES VACIOS POR COLUMNA")
display(revisar_vacios(recursos))

print("PRIMERAS 3 FILAS DESPUES DE LIMPIAR")
display(recursos[["codlocal", "numero_recurso", "tipo_recurso"] + CANTIDADES].head(3))


REGLAS DE COHERENCIA (cuantas filas incumplen cada regla)


,regla,filas_que_incumplen
0,La clave codlocal + numero_recurso esta repetida,0
1,La cantidad operativa es mayor que la cantidad...,0
2,Buen estado + reparacion no suma la cantidad o...,0
3,Menos de 3 anios + mas de 3 anios no suma la c...,0
4,Hay cantidades negativas,0
5,El tipo de recurso no se pudo traducir,0


VALORES VACIOS POR COLUMNA


,columna,vacios,porcentaje_vacios,con_dato
0,codlocal,0,0.00,126460
1,numero,0,0.00,126460
2,tipo_recurso_original,0,0.00,126460
3,cantidad_total,0,0.00,126460
4,cantidad_operativos,0,0.00,126460
5,cantidad_buen_estado,0,0.00,126460
6,cantidad_necesita_reparacion,0,0.00,126460
7,cantidad_menos_3_anios,0,0.00,126460
8,cantidad_mas_3_anios,0,0.00,126460
9,numero_recurso,0,0.00,126460


PRIMERAS 3 FILAS DESPUES DE LIMPIAR


,codlocal,numero_recurso,tipo_recurso,cantidad_total,cantidad_operativos,cantidad_buen_estado,cantidad_necesita_reparacion,cantidad_menos_3_anios,cantidad_mas_3_anios
0,000024,01,pc_escritorio,3,3,3,0,2,1
1,000024,09,proyector_multimedia,1,1,1,0,0,1
2,000038,01,pc_escritorio,4,4,4,0,0,4


### 4.3 Pivote a nivel local

`tipo_recurso` se abre en 13 columnas `rec_*` con la cantidad de cada equipo, y el `groupby` suma los totales por local. De 126,460 filas a una por `codlocal`.

In [7]:
recursos_por_tipo = recursos.pivot_table(
    index="codlocal", columns="tipo_recurso", values="cantidad_total",
    aggfunc="sum", fill_value=0, dropna=False,
)
recursos_por_tipo.columns = [f"rec_{c}" for c in recursos_por_tipo.columns]

recursos_local = recursos.groupby("codlocal").agg(
    recursos_total=("cantidad_total", "sum"),
    recursos_operativos=("cantidad_operativos", "sum"),
    recursos_buen_estado=("cantidad_buen_estado", "sum"),
    recursos_necesitan_reparacion=("cantidad_necesita_reparacion", "sum"),
    recursos_menos_3_anios=("cantidad_menos_3_anios", "sum"),
    recursos_mas_3_anios=("cantidad_mas_3_anios", "sum"),
    n_tipos_recurso=("tipo_recurso", "nunique"),
).join(recursos_por_tipo).reset_index()

recursos_local["pct_recursos_operativos"] = (
    100 * recursos_local["recursos_operativos"] / recursos_local["recursos_total"].replace(0, np.nan))
recursos_local["pct_operativos_reparacion"] = (
    100 * recursos_local["recursos_necesitan_reparacion"] / recursos_local["recursos_operativos"].replace(0, np.nan))
recursos_local["pct_operativos_mas_3_anios"] = (
    100 * recursos_local["recursos_mas_3_anios"] / recursos_local["recursos_operativos"].replace(0, np.nan))

assert recursos_local["codlocal"].is_unique
assert recursos["cantidad_total"].sum() == recursos_local["recursos_total"].sum()

print(f"ANTES:   {len(recursos):>8,} filas (una por recurso de cada local)")
print(f"DESPUES: {len(recursos_local):>8,} filas (una por local), {recursos_local.shape[1]} columnas")
print(f"\nCOLUMNAS NUEVAS QUE CREO EL PIVOTE ({len(recursos_por_tipo.columns)}):")
print("   " + ", ".join(recursos_por_tipo.columns))
print("\nPRIMERAS 3 FILAS")
display(recursos_local.head(3))


ANTES:    126,460 filas (una por recurso de cada local)
DESPUES:   42,221 filas (una por local), 24 columnas

COLUMNAS NUEVAS QUE CREO EL PIVOTE (13):
   rec_auriculares, rec_consola_audio, rec_laptop_convencional, rec_laptop_xo, rec_microservidor, rec_otro_recurso, rec_pc_escritorio, rec_pizarra_digital, rec_proyector_multimedia, rec_servidor, rec_tablet_aprendo_casa, rec_tablet_otros, rec_tablet_pronatel

PRIMERAS 3 FILAS


,codlocal,recursos_total,recursos_operativos,recursos_buen_estado,recursos_necesitan_reparacion,recursos_menos_3_anios,recursos_mas_3_anios,n_tipos_recurso,rec_auriculares,rec_consola_audio,rec_laptop_convencional,rec_laptop_xo,rec_microservidor,rec_otro_recurso,rec_pc_escritorio,rec_pizarra_digital,rec_proyector_multimedia,rec_servidor,rec_tablet_aprendo_casa,rec_tablet_otros,rec_tablet_pronatel,pct_recursos_operativos,pct_operativos_reparacion,pct_operativos_mas_3_anios
0,000024,4,4,4,0,2,2,2,0,0,0,0,0,0,3,0,1,0,0,0,0,100.00,0.00,50.00
1,000038,6,6,6,0,0,6,3,0,0,1,0,0,0,4,0,1,0,0,0,0,100.00,0.00,100.00
2,000043,1,1,1,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,100.00,0.00,100.00


## 5. Líneas de internet

Archivo `Loc_p3230_inter.dbf`. Varias filas por local, una por línea contratada.

### 5.1 Selección de columnas

11 de 16. Se eliminan los metadatos del envío y `p3230_1`, que es la pregunta filtro previa a los atributos de la línea.

In [8]:
COLUMNAS_LINEAS = [
    "codlocal",     # clave: local educativo
    "numero",       # clave: numero de linea dentro del local
    "p3230_2",      # la linea esta activa
    "p3230_3",      # medio de conexion
    "p3230_4",      # proveedor
    "p3230_5",      # quien la financia
    "p3230_6",      # ancho de banda contratado
    "p3230_7",      # porcentaje de velocidad garantizada
    "p3230_8",      # velocidad de bajada
    "p3230_9",      # velocidad de subida
    "p3230_10",     # tiene filtro web
]

lineas = seleccionar(lineas_raw, COLUMNAS_LINEAS, "Lineas de internet")

print("\nPRIMERAS 3 FILAS DESPUES DE SELECCIONAR")
display(lineas.head(3))


Lineas de internet: de 16 columnas nos quedamos con 11 y eliminamos 5

SE CONSERVAN (11):
   codlocal, numero, p3230_2, p3230_3, p3230_4, p3230_5, p3230_6, p3230_7, p3230_8, p3230_9, p3230_10

SE ELIMINAN (5):
   nroced, cuadro, p3230_1, fec_envio, anexo

PRIMERAS 3 FILAS DESPUES DE SELECCIONAR


,codlocal,numero,p3230_2,p3230_3,p3230_4,p3230_5,p3230_6,p3230_7,p3230_8,p3230_9,p3230_10
0,000024,1,1,02,01,01,6.00,70.00,6,6,2
1,000038,1,1,02,09,02,"1,000.00",100.00,50,100,1
2,000062,1,1,02,09,03,200.00,100.00,100,200,2


### 5.2 Limpieza

Renombrar, claves a texto, traducir medio y proveedor, métricas a número y apartar las 6 filas sin `codlocal` válido, que son las que producían los duplicados de clave del avance anterior.

In [9]:
lineas = lineas.rename(columns={
    "numero": "numero_linea",
    "p3230_2": "linea_activa_codigo",
    "p3230_3": "medio_codigo",
    "p3230_4": "proveedor_codigo",
    "p3230_5": "financia_codigo",
    "p3230_6": "ancho_contratado",
    "p3230_7": "velocidad_garantizada_pct",
    "p3230_8": "velocidad_bajada",
    "p3230_9": "velocidad_subida",
    "p3230_10": "filtro_web_codigo",
})

lineas["codlocal"] = cod(lineas["codlocal"], 6)
lineas["numero_linea"] = cod(lineas["numero_linea"], 1)

MEDIOS = {"01": "adsl", "02": "fibra_optica", "03": "hfc", "04": "cable_coaxial",
          "05": "satelital", "06": "wifi", "07": "usb_modem", "08": "radioenlace",
          "09": "otro_medio"}
PROVEEDORES = {"01": "movistar", "02": "claro", "03": "entel", "04": "bitel",
               "05": "vsat_minedu", "06": "level3", "07": "win", "08": "sencinet",
               "09": "otro_proveedor"}

lineas["medio"] = cod(lineas["medio_codigo"], 2).map(MEDIOS).fillna("sin_dato")
lineas["proveedor"] = cod(lineas["proveedor_codigo"], 2).map(PROVEEDORES).fillna("sin_dato")
lineas["linea_activa"] = cod(lineas["linea_activa_codigo"], 1).map({"1": True, "2": False}).astype("boolean")
lineas["filtro_web"] = cod(lineas["filtro_web_codigo"], 1).map({"1": True, "2": False}).astype("boolean")

METRICAS_LINEAS = ["ancho_contratado", "velocidad_garantizada_pct",
                   "velocidad_bajada", "velocidad_subida"]
for c in METRICAS_LINEAS:
    lineas[c] = pd.to_numeric(lineas[c], errors="coerce")

sin_codlocal = lineas["codlocal"].isna() | lineas["codlocal"].eq("000000")
lineas_invalidas = lineas[sin_codlocal].copy()
lineas = lineas[~sin_codlocal].copy()

print("REGLAS DE COHERENCIA (cuantas filas incumplen cada regla)")
calidad_lineas = pd.DataFrame({
    "regla": ["Filas apartadas por no tener codlocal valido",
              "La clave codlocal + numero_linea esta repetida",
              "El ancho contratado es negativo",
              "La velocidad garantizada esta fuera de 0 a 100",
              "La velocidad de bajada es negativa",
              "La velocidad de subida es negativa"],
    "filas_que_incumplen": [
        len(lineas_invalidas),
        lineas.duplicated(["codlocal", "numero_linea"]).sum(),
        (lineas["ancho_contratado"] < 0).sum(),
        ((lineas["velocidad_garantizada_pct"] < 0) | (lineas["velocidad_garantizada_pct"] > 100)).sum(),
        (lineas["velocidad_bajada"] < 0).sum(),
        (lineas["velocidad_subida"] < 0).sum()],
})
display(calidad_lineas)

print("VALORES VACIOS POR COLUMNA")
display(revisar_vacios(lineas))

print("PRIMERAS 3 FILAS DESPUES DE LIMPIAR")
display(lineas[["codlocal", "numero_linea", "medio", "proveedor", "linea_activa"] + METRICAS_LINEAS].head(3))


REGLAS DE COHERENCIA (cuantas filas incumplen cada regla)


,regla,filas_que_incumplen
0,Filas apartadas por no tener codlocal valido,6
1,La clave codlocal + numero_linea esta repetida,0
2,El ancho contratado es negativo,0
3,La velocidad garantizada esta fuera de 0 a 100,0
4,La velocidad de bajada es negativa,0
5,La velocidad de subida es negativa,0


VALORES VACIOS POR COLUMNA


,columna,vacios,porcentaje_vacios,con_dato
0,filtro_web,3107,10.81,25639
1,velocidad_garantizada_pct,2515,8.75,26231
2,ancho_contratado,2515,8.75,26231
3,linea_activa_codigo,0,0.00,28746
4,medio_codigo,0,0.00,28746
5,numero_linea,0,0.00,28746
6,codlocal,0,0.00,28746
7,financia_codigo,0,0.00,28746
8,proveedor_codigo,0,0.00,28746
9,velocidad_subida,0,0.00,28746


PRIMERAS 3 FILAS DESPUES DE LIMPIAR


,codlocal,numero_linea,medio,proveedor,linea_activa,ancho_contratado,velocidad_garantizada_pct,velocidad_bajada,velocidad_subida
0,000024,1,fibra_optica,movistar,True,6.00,70.00,6,6
1,000038,1,fibra_optica,otro_proveedor,True,"1,000.00",100.00,50,100
2,000062,1,fibra_optica,otro_proveedor,True,200.00,100.00,100,200


### 5.3 Pivote a nivel local

Se abren dos variables en columnas: `medio` en `lineas_medio_*` y `proveedor` en `lineas_proveedor_*`, con el número de líneas de cada tipo.

In [10]:
lineas_por_medio = lineas.pivot_table(
    index="codlocal", columns="medio", aggfunc="size", fill_value=0, dropna=False)
lineas_por_medio.columns = [f"lineas_medio_{c}" for c in lineas_por_medio.columns]

lineas_por_proveedor = lineas.pivot_table(
    index="codlocal", columns="proveedor", aggfunc="size", fill_value=0, dropna=False)
lineas_por_proveedor.columns = [f"lineas_proveedor_{c}" for c in lineas_por_proveedor.columns]

lineas_local = lineas.groupby("codlocal").agg(
    n_lineas=("numero_linea", "size"),
    n_lineas_activas=("linea_activa", lambda s: s.fillna(False).sum()),
    # verdadero si al menos una linea del colegio filtra contenido web
    tiene_filtro_contenido_web=("filtro_web", "max"),
    ancho_contratado_max=("ancho_contratado", "max"),
    velocidad_garantizada_mediana_pct=("velocidad_garantizada_pct", "median"),
    velocidad_bajada_max=("velocidad_bajada", "max"),
    velocidad_subida_max=("velocidad_subida", "max"),
).join([lineas_por_medio, lineas_por_proveedor]).reset_index()

assert lineas_local["codlocal"].is_unique
assert len(lineas) == lineas_local["n_lineas"].sum()

print(f"ANTES:   {len(lineas):>8,} filas (una por linea de cada local)")
print(f"DESPUES: {len(lineas_local):>8,} filas (una por local), {lineas_local.shape[1]} columnas")
print(f"\nCOLUMNAS NUEVAS POR MEDIO ({len(lineas_por_medio.columns)}):")
print("   " + ", ".join(lineas_por_medio.columns))
print(f"\nCOLUMNAS NUEVAS POR PROVEEDOR ({len(lineas_por_proveedor.columns)}):")
print("   " + ", ".join(lineas_por_proveedor.columns))
print("\nPRIMERAS 3 FILAS")
display(lineas_local.head(3))


ANTES:     28,746 filas (una por linea de cada local)
DESPUES:   24,752 filas (una por local), 28 columnas

COLUMNAS NUEVAS POR MEDIO (10):
   lineas_medio_adsl, lineas_medio_cable_coaxial, lineas_medio_fibra_optica, lineas_medio_hfc, lineas_medio_otro_medio, lineas_medio_radioenlace, lineas_medio_satelital, lineas_medio_sin_dato, lineas_medio_usb_modem, lineas_medio_wifi

COLUMNAS NUEVAS POR PROVEEDOR (10):
   lineas_proveedor_bitel, lineas_proveedor_claro, lineas_proveedor_entel, lineas_proveedor_level3, lineas_proveedor_movistar, lineas_proveedor_otro_proveedor, lineas_proveedor_sencinet, lineas_proveedor_sin_dato, lineas_proveedor_vsat_minedu, lineas_proveedor_win

PRIMERAS 3 FILAS


,codlocal,n_lineas,n_lineas_activas,tiene_filtro_contenido_web,ancho_contratado_max,velocidad_garantizada_mediana_pct,velocidad_bajada_max,velocidad_subida_max,lineas_medio_adsl,lineas_medio_cable_coaxial,lineas_medio_fibra_optica,lineas_medio_hfc,lineas_medio_otro_medio,lineas_medio_radioenlace,lineas_medio_satelital,lineas_medio_sin_dato,lineas_medio_usb_modem,lineas_medio_wifi,lineas_proveedor_bitel,lineas_proveedor_claro,lineas_proveedor_entel,lineas_proveedor_level3,lineas_proveedor_movistar,lineas_proveedor_otro_proveedor,lineas_proveedor_sencinet,lineas_proveedor_sin_dato,lineas_proveedor_vsat_minedu,lineas_proveedor_win
0,000019,1,1,<NA>,NaN,NaN,6,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1,000024,1,1,False,6.00,70.00,6,6,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,000038,1,1,True,"1,000.00",100.00,50,100,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0


## 6. Padrón de locales

Archivo `Padlocal.dbf`. Ya viene a una fila por local, así que no se pivotea.

### 6.1 Selección de columnas

17 de 19. Se eliminan `cloc_l1` y `tip_envio`.

`estado` y `d_estado` son columnas distintas (código y descripción). Se conservan las dos y se renombran a `estado_codigo` y `estado`: renombrar solo `d_estado` dejaría dos columnas con el mismo nombre.

In [11]:
COLUMNAS_PADRON = [
    "codlocal",      # clave con recursos y lineas
    "codcp_inei",    # clave con la cobertura movil
    "codccpp",       # codigo de centro poblado de MINEDU
    "cen_pob",       # nombre del centro poblado
    "dir_cen",       # direccion
    "localidad",     # localidad
    "codgeo",        # codigo de distrito
    "d_dpto",        # departamento
    "d_prov",        # provincia
    "d_dist",        # distrito
    "d_region",      # region DRE
    "codooii",       # codigo de UGEL
    "d_dreugel",     # nombre de la UGEL
    "estado",        # estado del local (codigo)
    "d_estado",      # estado del local (descripcion)
    "area_loc",      # ambito urbano o rural
    "ges_loc",       # tipo de gestion
]

padron = seleccionar(padron_raw, COLUMNAS_PADRON, "Padron de locales")

print("\nPRIMERAS 3 FILAS DESPUES DE SELECCIONAR")
display(padron.head(3))


Padron de locales: de 19 columnas nos quedamos con 17 y eliminamos 2

SE CONSERVAN (17):
   codlocal, codcp_inei, codccpp, cen_pob, dir_cen, localidad, codgeo, d_dpto, d_prov, d_dist, d_region, codooii, d_dreugel, estado, d_estado, area_loc, ges_loc

SE ELIMINAN (2):
   cloc_l1, tip_envio

PRIMERAS 3 FILAS DESPUES DE SELECCIONAR


,codlocal,codcp_inei,codccpp,cen_pob,dir_cen,localidad,codgeo,d_dpto,d_prov,d_dist,d_region,codooii,d_dreugel,estado,d_estado,area_loc,ges_loc
0,016100,0201050001,129688,CENTENARIO,JIRON TERESA GONZALES DE FANNY 543,NICRUPAMPA,020105,ANCASH,HUARAZ,INDEPENDENCIA,DRE ANCASH,020001,UGEL HUARAZ,1,Activo,1,A
1,015172,0201010001,114517,HUARUPAMPA,JIRON 28 DE JULIO S/N,HUARUPAMPA,020101,ANCASH,HUARAZ,HUARAZ,DRE ANCASH,020001,UGEL HUARAZ,1,Activo,1,A
2,015186,,610619,LA SOLEDAD,JIRON AMADEO FIGUEROA S/N,,020101,ANCASH,HUARAZ,HUARAZ,DRE ANCASH,020001,UGEL HUARAZ,1,Activo,1,A


### 6.2 Limpieza

Renombrar, códigos a texto con su ancho y traducir el ámbito (`1` urbano, `2` rural).

In [12]:
padron = padron.rename(columns={
    "codcp_inei": "cod_centro_poblado", "codccpp": "cod_ccpp_minedu",
    "cen_pob": "centro_poblado", "dir_cen": "direccion", "codgeo": "cod_distrito",
    "d_dpto": "departamento", "d_prov": "provincia", "d_dist": "distrito",
    "d_region": "region_dre", "codooii": "cod_ugel", "d_dreugel": "nombre_ugel",
    "estado": "estado_codigo", "d_estado": "estado",
    "area_loc": "area_codigo", "ges_loc": "gestion_codigo",
})

padron["codlocal"] = cod(padron["codlocal"], 6)
padron["cod_centro_poblado"] = cod(padron["cod_centro_poblado"], 10)
padron["cod_distrito"] = cod(padron["cod_distrito"], 6)
padron["cod_ugel"] = cod(padron["cod_ugel"], 6)

padron["area"] = padron["area_codigo"].map({"1": "Urbano", "2": "Rural"}).fillna("Sin dato")

assert not padron.columns.duplicated().any()
assert padron["codlocal"].is_unique

print(f"Padron limpio: {len(padron):,} locales, {padron.shape[1]} columnas")
sin_ccpp = padron["cod_centro_poblado"].isna().sum()
print(f"Locales sin codigo de centro poblado: {sin_ccpp:,} "
      f"({100 * sin_ccpp / len(padron):.1f} por ciento) -> no van a cruzar con la cobertura movil")

print("\nVALORES VACIOS POR COLUMNA")
display(revisar_vacios(padron))

print("PRIMERAS 3 FILAS DESPUES DE LIMPIAR")
display(padron[["codlocal", "departamento", "distrito", "area", "estado", "cod_centro_poblado"]].head(3))


Padron limpio: 69,642 locales, 18 columnas
Locales sin codigo de centro poblado: 6,664 (9.6 por ciento) -> no van a cruzar con la cobertura movil

VALORES VACIOS POR COLUMNA


,columna,vacios,porcentaje_vacios,con_dato
0,cod_centro_poblado,6664,9.57,62978
1,codlocal,0,0.00,69642
2,cod_ccpp_minedu,0,0.00,69642
3,centro_poblado,0,0.00,69642
4,direccion,0,0.00,69642
5,localidad,0,0.00,69642
6,cod_distrito,0,0.00,69642
7,departamento,0,0.00,69642
8,provincia,0,0.00,69642
9,distrito,0,0.00,69642


PRIMERAS 3 FILAS DESPUES DE LIMPIAR


,codlocal,departamento,distrito,area,estado,cod_centro_poblado
0,016100,ANCASH,INDEPENDENCIA,Urbano,Activo,0201050001
1,015172,ANCASH,HUARAZ,Urbano,Activo,0201010001
2,015186,ANCASH,HUARAZ,Urbano,Activo,<NA>


## 7. Cobertura móvil

Archivos de OSIPTEL. Una fila por centro poblado, tampoco se pivotea.

### 7.1 Selección de columnas

19 de 38. Se eliminan las 15 columnas `CG+CAR`, que suman cobertura referencial (red potencial, no servicio efectivo), y los nombres geográficos que ya vienen del padrón.

Los operadores del dataset son Bitel, Claro, Entel e Integratel. **Movistar no está**, así que la cobertura máxima queda subestimada donde Movistar es el operador principal.

In [13]:
# Las columnas de cobertura garantizada terminan en _cg. Las que terminan en _cg_mas_car quedan fuera
columnas_cobertura = [c for c in cobertura_raw.columns if c.endswith("_cg")]

COLUMNAS_COBERTURA = [
    "ubigeo",           # clave: codigo INEI del centro poblado
    "clasificacion",    # clasificacion del centro poblado
    "lat_y", "lon_x",   # coordenadas
] + columnas_cobertura

cobertura = seleccionar(cobertura_raw, COLUMNAS_COBERTURA, "Cobertura movil")

print(f"\nOPERADORES PRESENTES: {sorted({c.split('_')[0] for c in columnas_cobertura})}")
print("\nPRIMERAS 3 FILAS DESPUES DE SELECCIONAR")
display(cobertura.head(3))


Cobertura movil: de 38 columnas nos quedamos con 19 y eliminamos 19

SE CONSERVAN (19):
   ubigeo, clasificacion, lat_y, lon_x, bitel_3g_cg, bitel_4g_cg, bitel_5g_cg, claro_2g_cg, claro_3g_cg, claro_4g_cg, claro_5g_cg, entel_2g_cg, entel_3g_cg, entel_4g_cg, entel_5g_cg, integratel_2g_cg, integratel_3g_cg, integratel_4g_cg, integratel_5g_cg

SE ELIMINAN (19):
   departamento, provincia, distrito, centropoblado, bitel_3g_cg_mas_car, bitel_4g_cg_mas_car, bitel_5g_cg_mas_car, claro_2g_cg_mas_car, claro_3g_cg_mas_car, claro_4g_cg_mas_car, claro_5g_cg_mas_car, entel_2g_cg_mas_car, entel_3g_cg_mas_car, entel_4g_cg_mas_car, entel_5g_cg_mas_car, integratel_2g_cg_mas_car, integratel_3g_cg_mas_car, integratel_4g_cg_mas_car, integratel_5g_cg_mas_car

OPERADORES PRESENTES: ['bitel', 'claro', 'entel', 'integratel']

PRIMERAS 3 FILAS DESPUES DE SELECCIONAR


,ubigeo,clasificacion,lat_y,lon_x,bitel_3g_cg,bitel_4g_cg,bitel_5g_cg,claro_2g_cg,claro_3g_cg,claro_4g_cg,claro_5g_cg,entel_2g_cg,entel_3g_cg,entel_4g_cg,entel_5g_cg,integratel_2g_cg,integratel_3g_cg,integratel_4g_cg,integratel_5g_cg
0,0101010001,URBANO,-6.23,-77.87,0.55,0.46,0.41,0.61,0.37,0.21,0.00,0,0,0.01,0,0.35,0.84,0.99,0.00
1,0101010002,RURAL,-6.20,-77.90,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,0.00,0,0.00,0.00,0.00,0.00
2,0101010003,RURAL,-6.21,-77.87,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,0.00,0,0.00,0.00,1.00,0.00


### 7.2 Limpieza y variables nuevas

Clave a texto, cobertura en escala 0-100 y, por tecnología, la cobertura máxima entre operadores. Se usa el máximo y no el promedio porque lo relevante es que exista al menos un operador con servicio.

In [14]:
cobertura["cod_centro_poblado"] = cod(cobertura["ubigeo"], 10)

for c in columnas_cobertura:
    cobertura[c] = pd.to_numeric(cobertura[c], errors="coerce")
    if cobertura[c].max(skipna=True) <= 1:
        cobertura[c] = 100 * cobertura[c]

cobertura = cobertura.rename(columns={"clasificacion": "clasificacion_osiptel",
                                      "lat_y": "latitud_ccpp_osiptel",
                                      "lon_x": "longitud_ccpp_osiptel"})

cobertura_3g = [c for c in columnas_cobertura if "_3g_" in c]
cobertura_4g = [c for c in columnas_cobertura if "_4g_" in c]
cobertura_5g = [c for c in columnas_cobertura if "_5g_" in c]

cobertura["cobertura_3g_max_pct"] = cobertura[cobertura_3g].max(axis=1)
cobertura["cobertura_4g_max_pct"] = cobertura[cobertura_4g].max(axis=1)
cobertura["cobertura_5g_max_pct"] = cobertura[cobertura_5g].max(axis=1)
cobertura["operadores_4g_con_cobertura"] = (cobertura[cobertura_4g] > 0).sum(axis=1)

cobertura = cobertura.drop(columns=["ubigeo"])
assert cobertura["cod_centro_poblado"].is_unique

print(f"Cobertura limpia: {len(cobertura):,} centros poblados, {cobertura.shape[1]} columnas")
print(f"\nCOLUMNAS NUEVAS QUE SE CONSTRUYERON (4):")
print("   cobertura_3g_max_pct  = mayor cobertura 3G entre los operadores del centro poblado")
print("   cobertura_4g_max_pct  = mayor cobertura 4G entre los operadores del centro poblado")
print("   cobertura_5g_max_pct  = mayor cobertura 5G entre los operadores del centro poblado")
print("   operadores_4g_con_cobertura = cuantos operadores tienen algo de 4G ahi")

print("\nPRIMERAS 3 FILAS DESPUES DE LIMPIAR")
display(cobertura[["cod_centro_poblado", "clasificacion_osiptel", "cobertura_3g_max_pct",
                   "cobertura_4g_max_pct", "cobertura_5g_max_pct",
                   "operadores_4g_con_cobertura"]].head(3))


Cobertura limpia: 108,115 centros poblados, 23 columnas

COLUMNAS NUEVAS QUE SE CONSTRUYERON (4):
   cobertura_3g_max_pct  = mayor cobertura 3G entre los operadores del centro poblado
   cobertura_4g_max_pct  = mayor cobertura 4G entre los operadores del centro poblado
   cobertura_5g_max_pct  = mayor cobertura 5G entre los operadores del centro poblado
   operadores_4g_con_cobertura = cuantos operadores tienen algo de 4G ahi

PRIMERAS 3 FILAS DESPUES DE LIMPIAR


,cod_centro_poblado,clasificacion_osiptel,cobertura_3g_max_pct,cobertura_4g_max_pct,cobertura_5g_max_pct,operadores_4g_con_cobertura
0,0101010001,URBANO,84.07,99.21,41.00,4
1,0101010002,RURAL,0.00,0.00,0.00,0
2,0101010003,RURAL,0.00,100.00,0.00,1


## 8. Unión de las cuatro fuentes

Las cuatro tablas ya están a una fila por su clave. Se unen en tres pasos, todos `inner`:

| Paso | Qué une | Clave | Tipo |
|---|---|---|---|
| 1 | recursos con líneas | `codlocal` | `inner` |
| 2 | lo anterior con el padrón | `codlocal` | `inner` |
| 3 | lo anterior con la cobertura | `cod_centro_poblado` | `inner` (m:1) |

`inner` conserva únicamente las claves que están en las dos tablas. Aplicado tres veces, sobrevive
solo el local que aparece en las cuatro fuentes, así que la base final no tiene ningún faltante por
ausencia de fuente y entra directo al clustering sin imputar.

La unidad de análisis de la base final son esos locales con información completa, no los 69,642 del
padrón.

### 8.1 Cobertura de cada fuente sobre el padrón

Cuánto del padrón alcanza cada fuente antes de unir.

In [15]:
print("COBERTURA DE CADA FUENTE SOBRE EL PADRON COMPLETO")
print(f"Locales en el padron: {len(padron):,}\n")

disponibilidad = pd.DataFrame({
    "fuente": ["Recursos tecnologicos", "Lineas de internet", "Cobertura movil"],
    "locales_con_registro": [
        padron["codlocal"].isin(recursos_local["codlocal"]).sum(),
        padron["codlocal"].isin(lineas_local["codlocal"]).sum(),
        padron["cod_centro_poblado"].isin(cobertura["cod_centro_poblado"]).sum(),
    ],
})
disponibilidad["porcentaje_del_padron"] = 100 * disponibilidad["locales_con_registro"] / len(padron)
display(disponibilidad)

# Ningun local con datos puede quedar fuera del padron: se verifica que sea asi
huerfanos_recursos = (~recursos_local["codlocal"].isin(padron["codlocal"])).sum()
huerfanos_lineas = (~lineas_local["codlocal"].isin(padron["codlocal"])).sum()
print(f"Locales con recursos que NO estan en el padron: {huerfanos_recursos}")
print(f"Locales con lineas que NO estan en el padron:   {huerfanos_lineas}")


COBERTURA DE CADA FUENTE SOBRE EL PADRON COMPLETO
Locales en el padron: 69,642



,fuente,locales_con_registro,porcentaje_del_padron
0,Recursos tecnologicos,42221,60.63
1,Lineas de internet,24752,35.54
2,Cobertura movil,62724,90.07


Locales con recursos que NO estan en el padron: 0
Locales con lineas que NO estan en el padron:   0


### 8.2 La unión, paso a paso

In [16]:
datos_colegio = recursos_local.merge(lineas_local, on="codlocal",
                                     how="inner", validate="1:1")
print("PASO 1 — recursos con lineas (inner)")
print(f"   recursos:  {len(recursos_local):>7,} locales")
print(f"   lineas:    {len(lineas_local):>7,} locales")
print(f"   en ambos:  {len(datos_colegio):>7,} locales\n")

base = datos_colegio.merge(padron, on="codlocal", how="inner", validate="1:1")
print("PASO 2 — con el padron (inner)")
print(f"   entran:    {len(datos_colegio):>7,} locales")
print(f"   salen:     {len(base):>7,} locales")
print(f"   perdidos:  {len(datos_colegio) - len(base):>7,}\n")

antes_cobertura = len(base)
base = base.merge(cobertura, on="cod_centro_poblado", how="inner", validate="m:1")
print("PASO 3 — con la cobertura movil (inner, m:1)")
print(f"   entran:    {antes_cobertura:>7,} locales")
print(f"   salen:     {len(base):>7,} locales")
print(f"   perdidos:  {antes_cobertura - len(base):>7,}\n")

base["tiene_registro_recursos"] = True
base["tiene_registro_lineas"] = True
base["tiene_registro_cobertura"] = True

assert base["codlocal"].is_unique
assert base["recursos_total"].notna().all()
assert base["n_lineas"].notna().all()

print(f"BASE FINAL: {len(base):,} locales x {base.shape[1]} columnas")
print("Todos los locales tienen dato de las cuatro fuentes.")

print("\nPRIMERAS 3 FILAS DE LA BASE UNIDA")
display(base.head(3))


PASO 1 — recursos con lineas (inner)
   recursos:   42,221 locales
   lineas:     24,752 locales
   en ambos:   20,782 locales



PASO 2 — con el padron (inner)


   entran:     20,782 locales
   salen:      20,782 locales
   perdidos:        0

PASO 3 — con la cobertura movil (inner, m:1)
   entran:     20,782 locales
   salen:      18,483 locales
   perdidos:    2,299

BASE FINAL: 18,483 locales x 93 columnas
Todos los locales tienen dato de las cuatro fuentes.

PRIMERAS 3 FILAS DE LA BASE UNIDA


,codlocal,recursos_total,recursos_operativos,recursos_buen_estado,recursos_necesitan_reparacion,recursos_menos_3_anios,recursos_mas_3_anios,n_tipos_recurso,rec_auriculares,rec_consola_audio,rec_laptop_convencional,rec_laptop_xo,rec_microservidor,rec_otro_recurso,rec_pc_escritorio,rec_pizarra_digital,rec_proyector_multimedia,rec_servidor,rec_tablet_aprendo_casa,rec_tablet_otros,rec_tablet_pronatel,pct_recursos_operativos,pct_operativos_reparacion,pct_operativos_mas_3_anios,n_lineas,n_lineas_activas,tiene_filtro_contenido_web,ancho_contratado_max,velocidad_garantizada_mediana_pct,velocidad_bajada_max,velocidad_subida_max,lineas_medio_adsl,lineas_medio_cable_coaxial,lineas_medio_fibra_optica,lineas_medio_hfc,lineas_medio_otro_medio,lineas_medio_radioenlace,lineas_medio_satelital,lineas_medio_sin_dato,lineas_medio_usb_modem,lineas_medio_wifi,lineas_proveedor_bitel,lineas_proveedor_claro,lineas_proveedor_entel,lineas_proveedor_level3,lineas_proveedor_movistar,lineas_proveedor_otro_proveedor,lineas_proveedor_sencinet,lineas_proveedor_sin_dato,lineas_proveedor_vsat_minedu,lineas_proveedor_win,cod_centro_poblado,cod_ccpp_minedu,centro_poblado,direccion,localidad,cod_distrito,departamento,provincia,distrito,region_dre,cod_ugel,nombre_ugel,estado_codigo,estado,area_codigo,gestion_codigo,area,clasificacion_osiptel,latitud_ccpp_osiptel,longitud_ccpp_osiptel,bitel_3g_cg,bitel_4g_cg,bitel_5g_cg,claro_2g_cg,claro_3g_cg,claro_4g_cg,claro_5g_cg,entel_2g_cg,entel_3g_cg,entel_4g_cg,entel_5g_cg,integratel_2g_cg,integratel_3g_cg,integratel_4g_cg,integratel_5g_cg,cobertura_3g_max_pct,cobertura_4g_max_pct,cobertura_5g_max_pct,operadores_4g_con_cobertura,tiene_registro_recursos,tiene_registro_lineas,tiene_registro_cobertura
0,000024,4,4,4,0,2,2,2,0,0,0,0,0,0,3,0,1,0,0,0,0,100.00,0.00,50.00,1,1,False,6.00,70.00,6,6,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0101010001,131197,CHACHAPOYAS,JIRON JUNIN 697,,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,DRE AMAZONAS,010001,UGEL CHACHAPOYAS,1,Activo,1,A,Urbano,URBANO,-6.23,-77.87,55.05,46.07,41.00,60.82,37.04,20.67,0.00,0,0,1.17,0,34.75,84.07,99.21,0.00,84.07,99.21,41.00,4,True,True,True
1,000038,6,6,6,0,0,6,3,0,0,1,0,0,0,4,0,1,0,0,0,0,100.00,0.00,100.00,1,1,True,"1,000.00",100.00,50,100,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0101010001,131197,CHACHAPOYAS,JIRON PUNO 261,LUYA URCO,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,DRE AMAZONAS,010001,UGEL CHACHAPOYAS,1,Activo,1,A,Urbano,URBANO,-6.23,-77.87,55.05,46.07,41.00,60.82,37.04,20.67,0.00,0,0,1.17,0,34.75,84.07,99.21,0.00,84.07,99.21,41.00,4,True,True,True
2,000057,2,2,1,1,0,2,2,0,0,0,0,0,0,1,0,1,0,0,0,0,100.00,50.00,100.00,1,1,<NA>,NaN,NaN,6,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0101010001,131197,CHACHAPOYAS,JIRON AMAZONAS S/N,,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,DRE AMAZONAS,010001,UGEL CHACHAPOYAS,1,Activo,1,A,Urbano,URBANO,-6.23,-77.87,55.05,46.07,41.00,60.82,37.04,20.67,0.00,0,0,1.17,0,34.75,84.07,99.21,0.00,84.07,99.21,41.00,4,True,True,True


## 9. Revisión de la base unida

### 9.1 Valores vacíos por columna

In [17]:
print("VALORES VACIOS POR COLUMNA EN LA BASE UNIDA")
vacios_base = revisar_vacios(base)
display(vacios_base.head(30))

sin_vacios = (vacios_base["vacios"] == 0).sum()
print(f"Columnas sin ningun vacio: {sin_vacios} de {len(vacios_base)}")
vacios_base.to_csv(SALIDAS / "vacios_base.csv", index=False, encoding="utf-8-sig")


VALORES VACIOS POR COLUMNA EN LA BASE UNIDA


,columna,vacios,porcentaje_vacios,con_dato
0,tiene_filtro_contenido_web,2105,11.39,16378
1,ancho_contratado_max,1802,9.75,16681
2,velocidad_garantizada_mediana_pct,1802,9.75,16681
3,pct_operativos_reparacion,378,2.05,18105
4,pct_operativos_mas_3_anios,378,2.05,18105
5,recursos_menos_3_anios,0,0.00,18483
6,recursos_mas_3_anios,0,0.00,18483
7,n_tipos_recurso,0,0.00,18483
8,codlocal,0,0.00,18483
9,recursos_total,0,0.00,18483


Columnas sin ningun vacio: 88 de 93


### 9.2 Distribución de las variables que se van a modelar

`count` importa tanto como la media: indica sobre cuántos locales se calculó cada estadístico.

In [18]:
VARIABLES_ANALITICAS = [
    "recursos_total", "recursos_operativos", "recursos_necesitan_reparacion",
    "pct_recursos_operativos", "pct_operativos_reparacion", "pct_operativos_mas_3_anios",
    "n_lineas", "n_lineas_activas", "velocidad_bajada_max", "velocidad_subida_max",
    "cobertura_3g_max_pct", "cobertura_4g_max_pct", "cobertura_5g_max_pct",
    "operadores_4g_con_cobertura",
]

print("ESTADISTICOS DE LAS VARIABLES ANALITICAS")
resumen = base[VARIABLES_ANALITICAS].describe().T
resumen["porcentaje_vacios"] = 100 * base[VARIABLES_ANALITICAS].isna().mean().values
display(resumen)


ESTADISTICOS DE LAS VARIABLES ANALITICAS


,count,mean,std,min,25%,50%,75%,max,porcentaje_vacios
recursos_total,"18,483.00",76.60,138.51,1.00,9.00,30.00,90.00,"4,032.00",0.00
recursos_operativos,"18,483.00",66.98,125.47,0.00,7.00,26.00,76.00,"4,032.00",0.00
recursos_necesitan_reparacion,"18,483.00",12.54,49.71,0.00,0.00,0.00,6.00,"4,032.00",0.00
pct_recursos_operativos,"18,483.00",91.55,21.05,0.00,100.00,100.00,100.00,100.00,0.00
pct_operativos_reparacion,"18,105.00",15.38,27.62,0.00,0.00,0.00,19.63,100.00,2.05
pct_operativos_mas_3_anios,"18,105.00",61.57,41.80,0.00,11.63,82.35,100.00,100.00,2.05
n_lineas,"18,483.00",1.18,0.53,1.00,1.00,1.00,1.00,4.00,0.00
n_lineas_activas,"18,483.00",1.15,0.54,0.00,1.00,1.00,1.00,4.00,0.00
velocidad_bajada_max,"18,483.00",109.26,202.52,0.00,10.00,50.00,98.00,"2,000.00",0.00
velocidad_subida_max,"18,483.00",124.35,223.74,0.00,10.00,60.00,100.00,"2,000.00",0.00


### 9.3 Correlación entre esas variables

Dos variables muy correlacionadas miden casi lo mismo; si entran las dos al clustering esa dimensión pesa doble.

In [19]:
correlacion = base[VARIABLES_ANALITICAS].corr()

figura_corr = px.imshow(correlacion, text_auto=".2f", aspect="auto",
                        color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                        title="Correlacion entre las variables analiticas")
figura_corr.update_layout(height=650)
figura_corr.show()
figura_corr.write_html(SALIDAS / "figura_0_correlacion.html", include_plotlyjs="cdn")

pares = (correlacion.where(np.triu(np.ones(correlacion.shape), k=1).astype(bool))
         .stack().reset_index())
pares.columns = ["variable_1", "variable_2", "correlacion"]
altos = pares[pares["correlacion"].abs() >= 0.7].sort_values("correlacion", key=abs, ascending=False)

print("PARES DE VARIABLES QUE MIDEN CASI LO MISMO (correlacion mayor a 0.7)")
display(altos)


PARES DE VARIABLES QUE MIDEN CASI LO MISMO (correlacion mayor a 0.7)


,variable_1,variable_2,correlacion
63,n_lineas,n_lineas_activas,0.94
0,recursos_total,recursos_operativos,0.94
89,cobertura_4g_max_pct,operadores_4g_con_cobertura,0.89
76,velocidad_bajada_max,velocidad_subida_max,0.80
85,cobertura_3g_max_pct,cobertura_4g_max_pct,0.72


### 9.4 Exportación

In [20]:
INDICADORES = ["tiene_registro_recursos", "tiene_registro_lineas", "tiene_registro_cobertura"]

COLUMNAS_IDENTIFICACION = ["codlocal", "departamento", "provincia", "distrito", "region_dre",
                           "cod_ugel", "nombre_ugel", "centro_poblado", "cod_centro_poblado",
                           "area", "gestion_codigo"]

base_modelado = base[COLUMNAS_IDENTIFICACION + INDICADORES + VARIABLES_ANALITICAS].copy()

DESCRIPCIONES = {
    "codlocal": "Identificador unico del local educativo",
    "area": "Ambito del local: urbano o rural",
    "recursos_total": "Cantidad total de recursos tecnologicos del local",
    "recursos_operativos": "Cantidad de recursos que estan operativos",
    "recursos_necesitan_reparacion": "Equipos operativos que necesitan reparacion",
    "pct_recursos_operativos": "Porcentaje de los recursos totales que estan operativos",
    "pct_operativos_reparacion": "Porcentaje de los operativos que necesita reparacion",
    "pct_operativos_mas_3_anios": "Porcentaje de los operativos con mas de tres anios",
    "n_lineas": "Numero de lineas de internet registradas",
    "n_lineas_activas": "Numero de lineas declaradas activas",
    "velocidad_bajada_max": "Mayor velocidad de bajada entre las lineas del local",
    "velocidad_subida_max": "Mayor velocidad de subida entre las lineas del local",
    "cobertura_3g_max_pct": "Mayor cobertura 3G entre operadores del centro poblado",
    "cobertura_4g_max_pct": "Mayor cobertura 4G entre operadores del centro poblado",
    "cobertura_5g_max_pct": "Mayor cobertura 5G entre operadores del centro poblado",
    "operadores_4g_con_cobertura": "Operadores con algo de cobertura 4G en el centro poblado",
}

diccionario = pd.DataFrame({
    "variable": base_modelado.columns,
    "rol": ["identificacion" if c in COLUMNAS_IDENTIFICACION
            else "disponibilidad de fuente" if c in INDICADORES
            else "analitica" for c in base_modelado.columns],
    "descripcion": [DESCRIPCIONES.get(c, "") for c in base_modelado.columns],
    "vacios": [int(base_modelado[c].isna().sum()) for c in base_modelado.columns],
})

calidad = pd.concat([calidad_recursos.assign(fuente="Recursos tecnologicos"),
                     calidad_lineas.assign(fuente="Lineas de internet")],
                    ignore_index=True)[["fuente", "regla", "filas_que_incumplen"]]

base.to_csv(SALIDAS / "base_analitica_local.csv", index=False, encoding="utf-8-sig")
base_modelado.to_csv(SALIDAS / "base_modelado_preliminar.csv", index=False, encoding="utf-8-sig")
diccionario.to_csv(SALIDAS / "diccionario_variables.csv", index=False, encoding="utf-8-sig")
calidad.to_csv(SALIDAS / "calidad_fuentes.csv", index=False, encoding="utf-8-sig")

print("ARCHIVOS GUARDADOS EN salidas/")
for nombre in ["base_analitica_local.csv", "base_modelado_preliminar.csv",
               "diccionario_variables.csv", "calidad_fuentes.csv",
               "vacios_base.csv", "resultado_cruces.csv"]:
    print("   ", nombre)

print(f"\nbase_modelado: {len(base_modelado):,} filas x {base_modelado.shape[1]} columnas")
display(diccionario)


ARCHIVOS GUARDADOS EN salidas/
    base_analitica_local.csv
    base_modelado_preliminar.csv
    diccionario_variables.csv
    calidad_fuentes.csv
    vacios_base.csv
    resultado_cruces.csv

base_modelado: 18,483 filas x 28 columnas


,variable,rol,descripcion,vacios
0,codlocal,identificacion,Identificador unico del local educativo,0
1,departamento,identificacion,,0
2,provincia,identificacion,,0
3,distrito,identificacion,,0
4,region_dre,identificacion,,0
5,cod_ugel,identificacion,,0
6,nombre_ugel,identificacion,,0
7,centro_poblado,identificacion,,0
8,cod_centro_poblado,identificacion,,0
9,area,identificacion,Ambito del local: urbano o rural,0


## 10. Análisis exploratorio con Plotly

Cada figura declara en su título sobre qué universo se calcula. Las figuras 10.1 y 10.3 usan el padrón completo y el total de líneas registradas; el resto usa la base unida.

### 10.1 ¿Qué proporción del padrón está cubierta por cada fuente?

Sobre el padrón completo (69,642 locales). Es lo que explica por qué la base unida termina con menos.

In [21]:
fig1 = px.bar(
    disponibilidad, x="fuente", y="porcentaje_del_padron", text="porcentaje_del_padron",
    color="fuente", title="Disponibilidad de información por fuente (padrón completo)",
    labels={"porcentaje_del_padron": "% de locales", "fuente": "Fuente"},
)
fig1.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig1.update_layout(showlegend=False, yaxis_range=[0, 105])
fig1.show()
fig1.write_html(SALIDAS / "figura_1_disponibilidad_fuentes.html", include_plotlyjs="cdn")

vals = disponibilidad.set_index("fuente")["porcentaje_del_padron"]
display(Markdown(
    f"**Interpretación.** La cobertura movil alcanza {vals['Cobertura movil']:.1f}% del padrón, "
    f"los recursos {vals['Recursos tecnologicos']:.1f}% y las lineas {vals['Lineas de internet']:.1f}%. "
    "La principal limitación para el modelamiento no es el padrón, sino la disponibilidad desigual de las tablas temáticas."
))


**Interpretación.** La cobertura movil alcanza 90.1% del padrón, los recursos 60.6% y las lineas 35.5%. La principal limitación para el modelamiento no es el padrón, sino la disponibilidad desigual de las tablas temáticas.

### 10.2 ¿Difieren los recursos tecnológicos entre locales urbanos y rurales?

In [22]:
recursos_area = (
    base.loc[base["tiene_registro_recursos"]]
    .groupby("area", dropna=False)
    .agg(
        locales=("codlocal", "size"),
        mediana_recursos=("recursos_total", "median"),
        mediana_pct_operativos=("pct_recursos_operativos", "median"),
        mediana_pct_reparacion=("pct_operativos_reparacion", "median"),
    )
    .reset_index()
)
display(recursos_area)

fig2 = px.bar(
    recursos_area, x="area", y="mediana_recursos", color="area",
    text="mediana_recursos", title="Mediana de recursos registrados por ámbito",
    labels={"area": "Ámbito", "mediana_recursos": "Mediana de recursos"},
)
fig2.update_traces(textposition="outside")
fig2.update_layout(showlegend=False)
fig2.show()
fig2.write_html(SALIDAS / "figura_2_recursos_por_area.html", include_plotlyjs="cdn")

u = recursos_area.set_index("area")
if {"Urbano", "Rural"}.issubset(u.index):
    display(Markdown(
        f"**Interpretación.** Entre los locales con P91, la mediana es {u.loc['Urbano','mediana_recursos']:.0f} "
        f"recursos en el ámbito urbano y {u.loc['Rural','mediana_recursos']:.0f} en el rural. "
        f"La operatividad mediana es {u.loc['Urbano','mediana_pct_operativos']:.1f}% y "
        f"{u.loc['Rural','mediana_pct_operativos']:.1f}%, respectivamente. La comparación es descriptiva y no controla tamaño del local ni matrícula."
    ))


,area,locales,mediana_recursos,mediana_pct_operativos,mediana_pct_reparacion
0,Rural,6322,57.00,100.00,12.90
1,Urbano,12161,20.00,100.00,0.00


**Interpretación.** Entre los locales con P91, la mediana es 20 recursos en el ámbito urbano y 57 en el rural. La operatividad mediana es 100.0% y 100.0%, respectivamente. La comparación es descriptiva y no controla tamaño del local ni matrícula.

### 10.3 ¿Qué medios de conexión aparecen con mayor frecuencia?

Sobre las líneas registradas en el censo, no sobre los locales de la base: un local con tres líneas aporta tres observaciones.

In [23]:
medios = (
    lineas["medio"].value_counts(dropna=False)
    .rename_axis("medio").reset_index(name="lineas")
)
medios["pct_lineas"] = 100 * medios["lineas"] / medios["lineas"].sum()
fig3 = px.bar(
    medios.sort_values("lineas"), x="lineas", y="medio", orientation="h",
    text="pct_lineas", title="Líneas de internet registradas por medio (todas las líneas del censo)",
    labels={"lineas": "Número de líneas", "medio": "Medio"},
)
fig3.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig3.show()
fig3.write_html(SALIDAS / "figura_3_medios_internet.html", include_plotlyjs="cdn")

top = medios.iloc[0]
display(Markdown(
    f"**Interpretación.** {top['medio'].replace('_',' ')} concentra {top['pct_lineas']:.1f}% de las líneas registradas. "
    "La distribución describe las líneas presentes en P3230 y no a todos los locales del país."
))


**Interpretación.** fibra optica concentra 44.7% de las líneas registradas. La distribución describe las líneas presentes en P3230 y no a todos los locales del país.

### 10.4 ¿Cómo cambia la cobertura 4G territorial entre ámbitos?

In [24]:
cobertura_area = base.loc[
    base["tiene_registro_cobertura"] & base["cobertura_4g_max_pct"].notna(),
    ["codlocal", "area", "cobertura_4g_max_pct"],
]
fig4 = px.box(
    cobertura_area, x="area", y="cobertura_4g_max_pct", color="area",
    points=False, title="Máxima cobertura 4G garantizada entre operadores por ámbito",
    labels={"area": "Ámbito", "cobertura_4g_max_pct": "% del área del CCPP"},
)
fig4.update_layout(showlegend=False)
fig4.show()
fig4.write_html(SALIDAS / "figura_4_cobertura_4g_area.html", include_plotlyjs="cdn")

med4g = cobertura_area.groupby("area")["cobertura_4g_max_pct"].median()
texto = "; ".join(f"{a}: {v:.1f}%" for a, v in med4g.items())
display(Markdown(
    f"**Interpretación.** La mediana de la máxima cobertura 4G garantizada es {texto}. "
    "Es una medida del centro poblado: una cobertura alta no demuestra conectividad contratada ni calidad dentro del local."
))


**Interpretación.** La mediana de la máxima cobertura 4G garantizada es Rural: 1.2%; Urbano: 99.3%. Es una medida del centro poblado: una cobertura alta no demuestra conectividad contratada ni calidad dentro del local.

### 10.5 ¿Qué territorios combinan menor disponibilidad de recursos y menor cobertura 4G?

In [25]:
perfil_departamento = (
    base.groupby("departamento", dropna=False)
    .agg(
        locales=("codlocal", "size"),
        pct_con_p91=("tiene_registro_recursos", "mean"),
        pct_con_p3230=("tiene_registro_lineas", "mean"),
        mediana_recursos=("recursos_total", "median"),
        mediana_cobertura_4g=("cobertura_4g_max_pct", "median"),
    )
    .reset_index()
)
perfil_departamento[["pct_con_p91", "pct_con_p3230"]] *= 100

fig5 = px.scatter(
    perfil_departamento, x="mediana_recursos", y="mediana_cobertura_4g",
    size="locales", color="pct_con_p3230", hover_name="departamento",
    title="Perfil departamental: recursos registrados y cobertura 4G territorial",
    labels={
        "mediana_recursos": "Mediana de recursos (locales con P91)",
        "mediana_cobertura_4g": "Mediana de máxima cobertura 4G del CCPP (%)",
        "pct_con_p3230": "% de locales con P3230",
    },
    color_continuous_scale="Blues",
)
fig5.show()
fig5.write_html(SALIDAS / "figura_5_perfil_departamental.html", include_plotlyjs="cdn")

prioridad_descriptiva = perfil_departamento.dropna(
    subset=["mediana_recursos", "mediana_cobertura_4g"]
).sort_values(["mediana_cobertura_4g", "mediana_recursos"]).head(5)
display(prioridad_descriptiva[["departamento", "locales", "mediana_recursos", "mediana_cobertura_4g", "pct_con_p3230"]])
display(Markdown(
    "**Interpretación.** Los puntos hacia la esquina inferior izquierda combinan menor mediana de recursos registrados y menor cobertura 4G territorial. "
    "Son candidatos para revisión, no una clasificación definitiva: las medianas usan universos distintos cuando faltan P91 u OSIPTEL."
))


,departamento,locales,mediana_recursos,mediana_cobertura_4g,pct_con_p3230
8,HUANCAVELICA,605,104.00,14.96,100.00
18,PASCO,249,47.00,15.16,100.00
9,HUANUCO,616,28.00,18.59,100.00
5,CAJAMARCA,1044,36.00,21.53,100.00
2,APURIMAC,540,78.50,23.17,100.00


**Interpretación.** Los puntos hacia la esquina inferior izquierda combinan menor mediana de recursos registrados y menor cobertura 4G territorial. Son candidatos para revisión, no una clasificación definitiva: las medianas usan universos distintos cuando faltan P91 u OSIPTEL.

## 11. Conclusiones, limitaciones y siguiente etapa

In [26]:
resumen = {
    "locales_padron": int(len(base)),
    "columnas_base": int(base.shape[1]),
    "columnas_modelado_preliminar": int(base_modelado.shape[1]),
    "locales_p91": int(base["tiene_registro_recursos"].sum()),
    "locales_p3230": int(base["tiene_registro_lineas"].sum()),
    "locales_osiptel": int(base["tiene_registro_cobertura"].sum()),
    "pct_p91": float(100 * base["tiene_registro_recursos"].mean()),
    "pct_p3230": float(100 * base["tiene_registro_lineas"].mean()),
    "pct_osiptel": float(100 * base["tiene_registro_cobertura"].mean()),
    "filas_lineas_invalidas": int(len(lineas_invalidas)),
    "duplicados_codlocal_final": int(base.duplicated("codlocal").sum()),
}
(SALIDAS / "resumen_resultados.json").write_text(
    json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(resumen)


{'locales_padron': 18483,
 'columnas_base': 93,
 'columnas_modelado_preliminar': 28,
 'locales_p91': 18483,
 'locales_p3230': 18483,
 'locales_osiptel': 18483,
 'pct_p91': 100.0,
 'pct_p3230': 100.0,
 'pct_osiptel': 100.0,
 'filas_lineas_invalidas': 6,
 'duplicados_codlocal_final': 0}

### Conclusiones del avance

1. La integración funciona bien. Se mantiene una fila por `CODLOCAL` y los pivotes evitan que se dupliquen filas.
2. Hay que tener cuidado con la cobertura desigual entre fuentes. "Sin registro" no es lo mismo que "cero recursos" o "sin internet".
3. Cada fuente aporta algo distinto: P91 da la cantidad y operatividad de los equipos, P3230 las líneas declaradas, y OSIPTEL el contexto de cobertura territorial.
4. Las diferencias urbano-rurales y por departamento que se observaron son solo descriptivas. No permiten hablar de causalidad.

### Limitaciones

- No se dispone del valor de `P3230_2` cuando un local no tiene fila en P3230, por lo que esa ausencia no puede tratarse como "sin internet".
- La cobertura de OSIPTEL corresponde al centro poblado, no a una medición dentro del local educativo.
- No se incorporó matrícula, por lo que comparar cantidades absolutas puede reflejar solo el tamaño del local.
- Falta definir la estrategia de imputación y la selección final de variables antes de entrenar el clustering.

### Ruta para la segunda entrega

1. Seleccionar las variables analíticas no redundantes y definir una estrategia explícita para los faltantes.
2. Estandarizar las variables y usar PCA como apoyo visual, no como sustituto de la interpretación.
3. Probar distintas configuraciones de K-means y justificar la elección de `k` con elbow, silhouette y Davies-Bouldin.
4. Perfilar cada grupo según recursos, conectividad, ámbito y territorio, y validar su estabilidad y utilidad para la priorización.

---

**Archivos generados por este cuaderno**

- `salidas/base_analitica_local.csv`
- `salidas/base_modelado.csv`
- `salidas/diccionario_variables.csv`
- `salidas/calidad_fuentes.csv`
- `salidas/resultado_cruces.csv`
- `salidas/resumen_resultados.json`
- cinco visualizaciones interactivas en HTML